## Анализ метрик и когорт

In [2]:
import uuid
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "product_analytics"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

np.random.seed(42)
random.seed(42)

In [5]:
try:
    engine = create_engine(DATABASE_URL)

except Exception as e:
    print(f"\n Ошибка: {e}")

In [ ]:
'''
-- retention rate и churn rate
with session_dates as (
    select user_id, event_timestamp
    from events
    where event_name = 'session_start'
),
first_date as (
    select user_id, min(event_timestamp) as start_date
    from events
    where event_name = 'app_first_launch'
    group by user_id
),
metrics as (
    select 
        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '24 hours'
             and s.event_timestamp <  f.start_date + interval '48 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d1,

        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '168 hours'
             and s.event_timestamp <  f.start_date + interval '192 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d7,

        count(distinct case 
            when s.event_timestamp >= f.start_date + interval '720 hours'
             and s.event_timestamp <  f.start_date + interval '744 hours'
            then f.user_id 
        end)::float / nullif(count(distinct f.user_id), 0) as retention_rate_d30
    from first_date f
    left join session_dates s using(user_id)
)
select 
    retention_rate_d1,
    1 - retention_rate_d1 as churn_rate_d1,
    retention_rate_d7,
	1 - retention_rate_d7 as churn_rate_d7,
    retention_rate_d30,
	1 - retention_rate_d30 as churn_rate_d30
from metrics;
'''

In [ ]:
'''
-- воронка
select
count(distinct(case when event_name = 'onboarding_complete' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'app_first_launch' then user_id end)), 0) as conversion_rate_onboarding,
count(distinct(case when event_name = 'paywall_view' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'onboarding_complete' then user_id end)), 0) as conversion_rate_paywall,
count(distinct(case when event_name = 'subscription_purchase' then user_id end))::float /
	nullif(count(distinct(case when event_name = 'paywall_view' then user_id end)), 0) as conversion_rate_purchase
from events
'''

In [ ]:
'''
-- ARPU
select (SELECT sum(orders.amount) from orders where status = 'completed')::float 
	/ NULLIF((SELECT count(DISTINCT user_id) FROM users), 0) AS ARPU;
'''

In [ ]:
'''
-- ARPPU
select sum(amount)::float / count(distinct(user_id)) as ARPPU
from orders
where status = 'completed';
'''